In [1]:
!pip -q install transformers accelerate sentencepiece openpyxl pandas pypdf sentence-transformers langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb

import pandas as pd
import torch
import re

from google.colab import files
from pypdf import PdfReader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma



print("Questions(in).csv ve KVKK.pdf dosyalarını yükleyin")

uploaded = files.upload()



MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Model yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)


SYSTEM = """
Sen uzman bir hukuk asistanısın.

Türkçe cevap ver.

Kısa ve net cevap ver.

Emin değilsen bunu belirt.
"""


def ask(question, context=None):

    if context:

        user_content = f"""
KAYNAK METİN:

{context}

SORU:

{question}
"""

    else:

        user_content = question

    prompt = tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": SYSTEM
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    answer = pipe(prompt)[0]["generated_text"]

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        answer,
        flags=re.S
    )

    answer = answer.replace("\n", " ")
    answer = answer.replace("\r", " ")
    answer = " ".join(answer.split())

    return answer.strip()



print("KVKK PDF işleniyor...")

reader = PdfReader("KVKK.pdf")

docs = []

for i, page in enumerate(reader.pages):

    text = page.extract_text()

    if text and text.strip():

        docs.append(
            Document(
                page_content=text,
                metadata={"page": i + 1}
            )
        )

print("Sayfa sayısı:", len(docs))


splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64
)

chunks = splitter.split_documents(docs)

print("Chunk sayısı:", len(chunks))



print("Embedding oluşturuluyor...")

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

retriever = db.as_retriever(
    search_kwargs={"k": 2}
)



df = pd.read_csv(
    "Questions(in).csv",
    sep=";"
)

question_col = df.columns[0]

questions = (
    df[question_col]
    .dropna()
    .astype(str)
    .str.strip()
)

questions = questions[
    questions.str.len() > 5
]

print("Toplam soru:", len(questions))



results = []

for idx, question in enumerate(questions, start=1):

    print(f"Soru {idx}/{len(questions)}")



    baseline = ask(question)


    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        d.page_content
        for d in retrieved_docs
    )

    rag = ask(
        question,
        context=context
    )

    pages = ",".join(
        str(d.metadata.get("page"))
        for d in retrieved_docs
    )

    evidence = "\n\n====================\n\n".join(
        f"[Sayfa {d.metadata.get('page')}]\n{d.page_content[:1500]}"
        for d in retrieved_docs
    )

    results.append({
        "question": question,
        "baseline": baseline,
        "rag": rag,
        "retrieved_pages": pages,
        "retrieved_context": evidence
    })



result_df = pd.DataFrame(results)

display(result_df)

result_df.to_excel(
    "results_baseline_vs_rag.xlsx",
    index=False
)



readme = """
MODEL
-----
Qwen/Qwen2.5-1.5B-Instruct

INFERENCE
---------
Transformers Pipeline

PROMPT
------
Sen uzman bir hukuk asistanısın.

Türkçe cevap ver.

Kısa ve net cevap ver.

Emin değilsen bunu belirt.

GENERATION
----------
max_new_tokens = 128
do_sample = False
repetition_penalty = 1.1

RAG
---
Embedding Model : BAAI/bge-m3
Vector Database : ChromaDB
Chunk Size      : 512
Chunk Overlap   : 64
Top-K           : 2

KNOWLEDGE SOURCE
----------------
KVKK.pdf

EVALUATION
----------
Aynı soru seti hem baseline hem RAG için kullanılmıştır.
Aynı prompt ve aynı model kullanılmıştır.
Tek fark retrieval context'tir.
"""

with open(
    "README_experiment.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

print("\n✅ Tamamlandı")

files.download("results_baseline_vs_rag.xlsx")
files.download("README_experiment.txt")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requi

Saving KVKK.pdf to KVKK.pdf
Saving Questions(in).csv to Questions(in).csv
Model yükleniyor...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (29 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>